# Section 2.4 Synthetic Alpha Final Validation

This notebook validates the final clock-time synthetic alpha outputs before they are passed to section 2.5. It checks file/schema integrity, clock-time future-return construction, raw alpha calibration, alpha decay behavior, Brownian noise, formula consistency, and strategy input readiness.

It can be run from either the repository root or the `notebooks/` directory.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    if start.name == "notebooks":
        start = start.parent
    for candidate in [start] + list(start.parents):
        if (candidate / "src").exists() and ((candidate / "outputs").exists() or (candidate / ".git").exists()):
            return candidate
    raise RuntimeError("Could not find project root containing src/ and outputs/ or .git/")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ALPHA_DIR = PROJECT_ROOT / "outputs" / "alphas"
FIG_VALIDATION_DIR = ALPHA_DIR / "figures" / "validation"
FIG_VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

from src.alpha_validation import expected_alpha_paths, load_alpha_validation_inputs, run_final_alpha_validation

print("CURRENT_DIR:", Path.cwd().resolve())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ALPHA_DIR:", ALPHA_DIR)
print("Expected files:")
for name, path in expected_alpha_paths(ALPHA_DIR).items():
    print(f"  {name}: {path} exists={path.exists()}")

CURRENT_DIR: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/notebooks
PROJECT_ROOT: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project
ALPHA_DIR: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/alphas
Expected files:
  diagnostics: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/alphas/synthetic_alpha_diagnostics.csv exists=True
  baseline: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/alphas/synthetic_alpha_baseline_h5m_rho010.csv exists=True
  decay_diagnostics: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/alphas/synthetic_alpha_decay_diagnostics.csv exists=True
  decay_metadata: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project

## Load Inputs

In [2]:
data = load_alpha_validation_inputs(ALPHA_DIR)
diagnostics = data["diagnostics"]
baseline_alpha = data["baseline"]
decay_diagnostics = data["decay_diagnostics"]
decay_metadata = data["decay_metadata"]

print("diagnostics:", diagnostics.shape)
print("baseline_alpha:", baseline_alpha.shape)
print("decay_diagnostics:", decay_diagnostics.shape)
print("decay_metadata:", decay_metadata.shape)

display(diagnostics.head())
display(baseline_alpha.head())
display(decay_diagnostics)
display(decay_metadata)

diagnostics: (15, 20)
baseline_alpha: (200000, 26)
decay_diagnostics: (4, 13)
decay_metadata: (4, 2)


,horizon_minutes,target_corr,n_rows_total,n_valid_future_return,fraction_missing_future_return,median_future_time_gap_seconds,p95_future_time_gap_seconds,max_future_time_gap_seconds,var_future_return,mean_inv_price_sq,alpha_x,alpha_y,empirical_corr_alpha_return,alpha_mean,alpha_std,future_return_mean,future_return_std,realized_unbiased_slope,corr_abs_error,recommended_baseline_flag
0,1.0,0.05,200000,199314,0.00343,0.0,0.0,30.0,0.000001,0.000405,0.0025,0.002997,0.048913,-1.768561e-07,0.000060,-0.000005,0.001208,0.983178,0.001087,False
1,1.0,0.10,200000,199314,0.00343,0.0,0.0,30.0,0.000001,0.000405,0.0100,0.005971,0.099832,-2.007611e-07,0.000122,-0.000005,0.001208,0.989277,0.000168,False
2,1.0,0.20,200000,199314,0.00343,0.0,0.0,30.0,0.000001,0.000405,0.0400,0.011760,0.204099,1.918840e-07,0.000242,-0.000005,0.001208,1.017644,0.004099,False
3,1.0,0.30,200000,199314,0.00343,0.0,0.0,30.0,0.000001,0.000405,0.0900,0.017174,0.300241,-1.330812e-07,0.000363,-0.000005,0.001208,1.000223,0.000241,False
4,1.0,0.50,200000,199314,0.00343,0.0,0.0,30.0,0.000001,0.000405,0.2500,0.025986,0.497278,4.071764e-07,0.000604,-0.000005,0.001208,0.994600,0.002722,False


,date,time,stock,mid,midEnd,spread,depth,timestamp,target_timestamp_h,future_timestamp_h,...,alpha_y,target_corr,horizon_minutes,valid_future_return,delta_w,source_file,alpha_state_H1m,alpha_state_H5m,alpha_state_H30m,alpha_state_H60m
0,2019-01-02,09:30:10,A,66.215,66.220,0.285,300.0,2019-01-02 09:30:10,2019-01-02 09:35:10,2019-01-02 09:35:10,...,0.005701,0.1,5.0,True,-0.913528,bin201901.csv,-0.000048,-0.000048,-0.000048,-0.000048
1,2019-01-02,09:30:20,A,66.335,66.335,0.145,350.0,2019-01-02 09:30:20,2019-01-02 09:35:20,2019-01-02 09:35:20,...,0.005701,0.1,5.0,True,-1.513881,bin201901.csv,-0.000056,-0.000050,-0.000049,-0.000049
2,2019-01-02,09:30:30,A,66.335,66.335,0.145,600.0,2019-01-02 09:30:30,2019-01-02 09:35:30,2019-01-02 09:35:30,...,0.005701,0.1,5.0,True,-1.611917,bin201901.csv,-0.000063,-0.000052,-0.000049,-0.000049
3,2019-01-02,09:30:40,A,66.340,66.340,0.140,500.0,2019-01-02 09:30:40,2019-01-02 09:35:40,2019-01-02 09:35:40,...,0.005701,0.1,5.0,True,-2.978249,bin201901.csv,-0.000082,-0.000056,-0.000050,-0.000049
4,2019-01-02,09:31:00,A,66.270,66.270,0.070,375.0,2019-01-02 09:31:00,2019-01-02 09:36:00,2019-01-02 09:36:00,...,0.005701,0.1,5.0,True,1.426089,bin201901.csv,-0.000033,-0.000046,-0.000048,-0.000048


,half_life_minutes,output_col,n_valid,corr_raw_alpha_future_return,corr_alpha_state_future_return,raw_alpha_std,alpha_state_std,raw_alpha_std_bps,alpha_state_std_bps,state_to_raw_std_ratio,mean_abs_raw_alpha,mean_abs_alpha_state,state_to_raw_mean_abs_ratio
0,1.0,alpha_state_H1m,197168,0.099555,0.277577,0.000259,0.000068,2.585011,0.678670,0.262540,0.000158,0.000044,0.276495
1,5.0,alpha_state_H5m,197168,0.099555,0.205328,0.000259,0.000039,2.585011,0.389400,0.150638,0.000158,0.000023,0.145135
2,30.0,alpha_state_H30m,197168,0.099555,0.041326,0.000259,0.000058,2.585011,0.578769,0.223894,0.000158,0.000022,0.138124
3,60.0,alpha_state_H60m,197168,0.099555,0.023665,0.000259,0.000081,2.585011,0.805800,0.311720,0.000158,0.000034,0.213334


,half_life_minutes,output_col
0,1.0,alpha_state_H1m
1,5.0,alpha_state_H5m
2,30.0,alpha_state_H30m
3,60.0,alpha_state_H60m


## Run Final Validation

In [3]:
validation = run_final_alpha_validation(ALPHA_DIR)

results = validation["results"]
clean_alpha_table = validation["clean_alpha_table"]
clean_decay_table = validation["clean_decay_table"]

display(results)
display(clean_alpha_table)
display(clean_decay_table)

,name,passed,details
0,file/schema,True,"All required files, columns, scenarios, and me..."
1,clock-time horizons,True,Clock-time horizon construction passed.
2,raw alpha calibration,True,All h/rho calibration checks passed.
3,alpha decay,True,warning: alpha_state_H60m std exceeds alpha_st...
4,Brownian noise,True,Brownian noise checks passed.
5,formula consistency,True,Formula reconstruction matches alpha_synthetic.
6,strategy input readiness,True,Saved /Users/ulysse/Desktop/Studies/Imperial/C...


,h_minutes,rho_target,corr_empirical,corr_error,unbiased_slope,alpha_std_bps,future_return_std_bps,missing_pct
0,1.0,0.05,0.0489,0.0011,0.9832,0.601,12.078,0.343
1,1.0,0.10,0.0998,0.0002,0.9893,1.219,12.078,0.343
2,1.0,0.20,0.2041,0.0041,1.0176,2.422,12.078,0.343
3,1.0,0.30,0.3002,0.0002,1.0002,3.625,12.078,0.343
4,1.0,0.50,0.4973,0.0027,0.9946,6.039,12.078,0.343
5,5.0,0.05,0.0487,0.0013,0.9760,1.287,25.789,1.416
6,5.0,0.10,0.0996,0.0004,0.9932,2.585,25.789,1.416
7,5.0,0.20,0.1990,0.0010,0.9979,5.143,25.789,1.416
8,5.0,0.30,0.3054,0.0054,1.0232,7.698,25.789,1.416
9,5.0,0.50,0.5017,0.0017,1.0069,12.851,25.789,1.416


,H_minutes,output_col,corr_raw_alpha_future_return,corr_alpha_state_future_return,raw_alpha_std_bps,alpha_state_std_bps,state_to_raw_std_ratio
0,1.0,alpha_state_H1m,0.0996,0.2776,2.585,0.679,0.2625
1,5.0,alpha_state_H5m,0.0996,0.2053,2.585,0.389,0.1506
2,30.0,alpha_state_H30m,0.0996,0.0413,2.585,0.579,0.2239
3,60.0,alpha_state_H60m,0.0996,0.0237,2.585,0.806,0.3117


## Clock-Time Horizon Checks

In [4]:
display(validation["gap_summary"])

valid = baseline_alpha.loc[baseline_alpha["valid_future_return"].astype(bool)].copy()
print("valid rows:", len(valid))
print("fraction missing future return:", 1.0 - len(valid) / len(baseline_alpha))
print("future_time_gap_seconds max:", valid["future_time_gap_seconds"].max())
print("future_timestamp >= target_timestamp:", bool((valid["future_timestamp_h"] >= valid["target_timestamp_h"]).all()))

,future_time_gap_seconds
count,197168.000000
mean,0.476852
std,2.559022
min,0.000000
50%,0.000000
95%,0.000000
99%,10.000000
max,30.000000


valid rows: 197168
fraction missing future return: 0.01415999999999995
future_time_gap_seconds max: 30.0
future_timestamp >= target_timestamp: True


## Raw Alpha Calibration Checks

In [5]:
failures = validation["calibration_failures"]
if failures.empty:
    print("All calibration rows pass the strict report thresholds.")
else:
    print("Calibration rows failing thresholds:")
    display(failures)

display(clean_alpha_table)

All calibration rows pass the strict report thresholds.


,h_minutes,rho_target,corr_empirical,corr_error,unbiased_slope,alpha_std_bps,future_return_std_bps,missing_pct
0,1.0,0.05,0.0489,0.0011,0.9832,0.601,12.078,0.343
1,1.0,0.10,0.0998,0.0002,0.9893,1.219,12.078,0.343
2,1.0,0.20,0.2041,0.0041,1.0176,2.422,12.078,0.343
3,1.0,0.30,0.3002,0.0002,1.0002,3.625,12.078,0.343
4,1.0,0.50,0.4973,0.0027,0.9946,6.039,12.078,0.343
5,5.0,0.05,0.0487,0.0013,0.9760,1.287,25.789,1.416
6,5.0,0.10,0.0996,0.0004,0.9932,2.585,25.789,1.416
7,5.0,0.20,0.1990,0.0010,0.9979,5.143,25.789,1.416
8,5.0,0.30,0.3054,0.0054,1.0232,7.698,25.789,1.416
9,5.0,0.50,0.5017,0.0017,1.0069,12.851,25.789,1.416


## Alpha Decay Checks

In [6]:
display(clean_decay_table)
print("baseline_alpha_for_strategy equals alpha_state_H5m:", np.allclose(
    baseline_alpha["baseline_alpha_for_strategy"], baseline_alpha["alpha_state_H5m"]
))
print("state columns have NaNs:")
for col in ["alpha_state_H1m", "alpha_state_H5m", "alpha_state_H30m", "alpha_state_H60m"]:
    print(f"  {col}: {baseline_alpha[col].isna().sum()} NaNs, std={baseline_alpha[col].std():.6g}")

,H_minutes,output_col,corr_raw_alpha_future_return,corr_alpha_state_future_return,raw_alpha_std_bps,alpha_state_std_bps,state_to_raw_std_ratio
0,1.0,alpha_state_H1m,0.0996,0.2776,2.585,0.679,0.2625
1,5.0,alpha_state_H5m,0.0996,0.2053,2.585,0.389,0.1506
2,30.0,alpha_state_H30m,0.0996,0.0413,2.585,0.579,0.2239
3,60.0,alpha_state_H60m,0.0996,0.0237,2.585,0.806,0.3117


baseline_alpha_for_strategy equals alpha_state_H5m: True
state columns have NaNs:
  alpha_state_H1m: 0 NaNs, std=6.74819e-05
  alpha_state_H5m: 0 NaNs, std=3.87679e-05
  alpha_state_H30m: 0 NaNs, std=5.74935e-05
  alpha_state_H60m: 0 NaNs, std=8.00337e-05


## Brownian Noise and Formula Checks

In [7]:
display(validation["brownian_summary"])
display(validation["formula_summary"])

valid = baseline_alpha.loc[baseline_alpha["valid_future_return"].astype(bool)].copy()
alpha_reconstructed = valid["alpha_x"] * valid["future_return_h"] + valid["alpha_y"] * valid["delta_w"] / valid["mid"]
error = (alpha_reconstructed - valid["alpha_synthetic"]).abs()
print("formula max abs error:", error.max())
print("formula mean abs error:", error.mean())

,delta_w_mean,delta_w_var,expected_delta_w_var
0,-0.000917,5.01565,5.0


,max_abs_error,mean_abs_error
0,9.931292e-17,2.293393e-17


formula max abs error: 9.93129189996722e-17
formula mean abs error: 2.2933930895402673e-17


## Strategy Input for Section 2.5

In [8]:
strategy_path = ALPHA_DIR / "strategy_alpha_input_h5m_rho010_H5m.csv"
print("strategy input path:", strategy_path)
print("exists:", strategy_path.exists())
display(validation["strategy_input_head"])

if strategy_path.exists():
    strategy_sample = pd.read_csv(strategy_path, nrows=5)
    display(strategy_sample)

strategy input path: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/alphas/strategy_alpha_input_h5m_rho010_H5m.csv
exists: True


,date,time,timestamp,stock,mid,alpha_raw,alpha_for_strategy,future_return_h,valid_future_return,spread,depth,source_file
0,2019-01-02,09:30:10,2019-01-02 09:30:10,A,66.215,-0.000048,-0.000048,0.003020,True,0.285,300.0,bin201901.csv
1,2019-01-02,09:30:20,2019-01-02 09:30:20,A,66.335,-0.000114,-0.000050,0.001583,True,0.145,350.0,bin201901.csv
2,2019-01-02,09:30:30,2019-01-02 09:30:30,A,66.335,-0.000123,-0.000052,0.001583,True,0.145,600.0,bin201901.csv
3,2019-01-02,09:30:40,2019-01-02 09:30:40,A,66.340,-0.000239,-0.000056,0.001733,True,0.140,500.0,bin201901.csv
4,2019-01-02,09:31:00,2019-01-02 09:31:00,A,66.270,0.000156,-0.000046,0.003320,True,0.070,375.0,bin201901.csv


,date,time,timestamp,stock,mid,alpha_raw,alpha_for_strategy,future_return_h,valid_future_return,spread,depth,source_file
0,2019-01-02,09:30:10,2019-01-02 09:30:10,A,66.215,-0.000048,-0.000048,0.003020,True,0.285,300.0,bin201901.csv
1,2019-01-02,09:30:20,2019-01-02 09:30:20,A,66.335,-0.000114,-0.000050,0.001583,True,0.145,350.0,bin201901.csv
2,2019-01-02,09:30:30,2019-01-02 09:30:30,A,66.335,-0.000123,-0.000052,0.001583,True,0.145,600.0,bin201901.csv
3,2019-01-02,09:30:40,2019-01-02 09:30:40,A,66.340,-0.000239,-0.000056,0.001733,True,0.140,500.0,bin201901.csv
4,2019-01-02,09:31:00,2019-01-02 09:31:00,A,66.270,0.000156,-0.000046,0.003320,True,0.070,375.0,bin201901.csv


## Final Summary

In [9]:
summary_path = ALPHA_DIR / "final_section_2_4_validation_summary.txt"
print(summary_path.read_text())
print("Validation figures saved to:", FIG_VALIDATION_DIR)
for path in sorted(FIG_VALIDATION_DIR.glob("*.png")):
    print(" -", path.relative_to(PROJECT_ROOT))

Final Section 2.4 Validation Summary

Check status:
- file/schema: PASS - All required files, columns, scenarios, and metadata found.
- clock-time horizons: PASS - Clock-time horizon construction passed.
- raw alpha calibration: PASS - All h/rho calibration checks passed.
- alpha decay: PASS - warning: alpha_state_H60m std exceeds alpha_state_H1m std
- Brownian noise: PASS - Brownian noise checks passed.
- formula consistency: PASS - Formula reconstruction matches alpha_synthetic.
- strategy input readiness: PASS - Saved /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/alphas/strategy_alpha_input_h5m_rho010_H5m.csv

Baseline values:
h = 5 minutes
rho = 0.10
H = 5 minutes
empirical corr = 0.099555
corr error = 0.000445
unbiased slope = 0.993203
future return std bps = 25.789
alpha std bps = 2.585
fraction missing future return = 0.014160
corr alpha_state_H5m vs future return = 0.205328
alpha_state_H5m std bps = 0.389

Conclusion:
Sec